In [2]:
import sys
print(sys.executable)

/Users/zaamine/Desktop/Amine/Training/Kaggle/Corporacion Favorita/retail-demand-forecasting-platform-aws/.venv/bin/python


In [3]:
import pandas as pd
print(pd.__version__)

3.0.6


# Business context

The project simulates a short-term retail demand-planning use case.

The objective is to produce daily sales forecasts for each `store × product family` series over the next 14 days. Forecasts are refreshed daily once the latest observed sales data is available.

The forecasts are intended to support short-term inventory and replenishment planning by helping anticipate demand peaks while reducing both stock-out and overstock risk.

Because the dataset contains sales but no inventory or stock-out information, observed sales are treated as a proxy for customer demand. A production retailer use case would require inventory availability data to distinguish low demand from constrained sales.

### Forecasting setup

- **Target:** daily sales
- **Granularity:** `store × family × day`
- **Forecast horizon:** 14 days
- **Refresh cadence:** daily
- **Forecast origin:** end of day `t`, using information available up to that point
- **Primary business use case:** short-term inventory and replenishment planning
- **Primary evaluation metric:** WAPE
- **Additional diagnostics:** MAE, RMSE and forecast bias

All model features must respect point-in-time availability at the forecast origin.

A separate promotion-analysis layer will later estimate counterfactual promotional uplift to support promotion-planning decisions.

## Raw Data Exploration

Initial inspection of the Corporación Favorita — Store Sales: Time Series Forecasting dataset.

Objectives:
- inspect raw schemas and data types;
- confirm the business grain of the sales table;
- validate store, family and series cardinalities;
- inspect date coverage and missing values;
- understand how the auxiliary tables relate to the forecasting problem.

No data transformation is performed in this notebook.

In [4]:
from pathlib import Path

# Get the notebook's current working directory.
current_path = Path.cwd()

# Walk up the directory tree until the project root is found.
for path in [current_path, *current_path.parents]:
    if (path / "pyproject.toml").exists():
        PROJECT_ROOT = path
        break
else:
    raise RuntimeError("Project root not found.")

# Paths to the raw Favorita datasets.
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "favorita"

TRAIN_PATH = DATA_DIR / "train" / "train.csv"
STORES_PATH = DATA_DIR / "stores" / "stores.csv"
OIL_PATH = DATA_DIR / "oil" / "oil.csv"
HOLIDAYS_PATH = DATA_DIR / "holidays_events" / "holidays_events.csv"
TRANSACTIONS_PATH = DATA_DIR / "transactions" / "transactions.csv"

# Quick path sanity check.
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("train.csv exists:", TRAIN_PATH.exists())

Project root: /Users/zaamine/Desktop/Amine/Training/Kaggle/Corporacion Favorita/retail-demand-forecasting-platform-aws
Data directory: /Users/zaamine/Desktop/Amine/Training/Kaggle/Corporacion Favorita/retail-demand-forecasting-platform-aws/data/raw/favorita
train.csv exists: True


### `train.csv`:
The training data, comprising time series of features store_nbr, family, and onpromotion as well as the target sales.
- store_nbr identifies the store at which the products are sold.
- family identifies the type of product sold.
- sales gives the total sales for a product family at a particular store at a given date. Fractional values are possible since products can be sold in fractional units (1.5 kg of cheese, for instance, as opposed to 1 bag of chips).
- onpromotion gives the total number of items in a product family that were being promoted at a store at a given date.

Additional notes:
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

In [5]:
train = pd.read_csv(
    TRAIN_PATH,
    parse_dates=["date"]
)

train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [5]:
print(train.head().to_string())

   id       date  store_nbr      family  sales  onpromotion
0   0 2013-01-01          1  AUTOMOTIVE    0.0            0
1   1 2013-01-01          1   BABY CARE    0.0            0
2   2 2013-01-01          1      BEAUTY    0.0            0
3   3 2013-01-01          1   BEVERAGES    0.0            0
4   4 2013-01-01          1       BOOKS    0.0            0


In [6]:
# Inspect the dataset dimensions and column types.
print(f"Rows: {len(train):,}")
print(f"Columns: {train.shape[1]}")

display(train.dtypes.to_frame("dtype"))

Rows: 3,000,888
Columns: 6


,dtype
id,int64
date,datetime64[us]
store_nbr,int64
family,str
sales,float64
onpromotion,int64


In [7]:
# Inspect the time coverage of the sales history.
start_date = train["date"].min()
end_date = train["date"].max()
n_calendar_days = (end_date - start_date).days + 1

print("Start date:", start_date.date())
print("End date:", end_date.date())
print("Calendar days:", n_calendar_days)

Start date: 2013-01-01
End date: 2017-08-15
Calendar days: 1688


In [8]:
# Inspect the dimensions of the store-family hierarchy.
n_stores = train["store_nbr"].nunique()
n_families = train["family"].nunique()

# Each store-family pair defines one demand time series.
n_series = (
    train[["store_nbr", "family"]]
    .drop_duplicates()
    .shape[0]
)

print("Stores:", n_stores)
print("Families:", n_families)
print("Potential store-family combinations:", n_stores * n_families)
print("Observed store-family series:", n_series)

Stores: 54
Families: 33
Potential store-family combinations: 1782
Observed store-family series: 1782


In [9]:
# Check whether date-store-family uniquely identifies each observation.
grain = ["date", "store_nbr", "family"]

duplicate_rows = train.duplicated(grain).sum()

print("Duplicate date-store-family rows:", duplicate_rows)

Duplicate date-store-family rows: 0


In [6]:
# Identify product-family labels containing commas, which require quote-aware CSV parsing.
families_with_commas = (
    train.loc[
        train["family"].str.contains(",", regex=False),
        "family",
    ]
    .value_counts()
)

display(families_with_commas)

family
LIQUOR,WINE,BEER    90936
Name: count, dtype: int64

In [7]:
# Inspect raw rows for the comma-containing product family.
display(
    train.loc[
        train["family"] == "LIQUOR,WINE,BEER"
    ].head()
)

,id,date,store_nbr,family,sales,onpromotion
22,22,2013-01-01,1,"LIQUOR,WINE,BEER",0.0,0
55,55,2013-01-01,10,"LIQUOR,WINE,BEER",0.0,0
88,88,2013-01-01,11,"LIQUOR,WINE,BEER",0.0,0
121,121,2013-01-01,12,"LIQUOR,WINE,BEER",0.0,0
154,154,2013-01-01,13,"LIQUOR,WINE,BEER",0.0,0


In [10]:
# Check how many distinct dates are actually present in the dataset.
n_observed_dates = train["date"].nunique()

print("Calendar days:", n_calendar_days)
print("Observed dates:", n_observed_dates)
print("Missing calendar dates:", n_calendar_days - n_observed_dates)

Calendar days: 1688
Observed dates: 1684
Missing calendar dates: 4


In [11]:
# Build the complete daily calendar and identify missing dates.
full_calendar = pd.date_range(
    start=train["date"].min(),
    end=train["date"].max(),
    freq="D"
)

missing_dates = full_calendar.difference(train["date"].unique())

print("Missing dates:")
for date in missing_dates:
    print(date.date())

Missing dates:
2013-12-25
2014-12-25
2015-12-25
2016-12-25


In [12]:
# Inspect missing values in each column.
missing_values = train.isna().sum().to_frame("missing_count")
missing_values["missing_pct"] = (
    missing_values["missing_count"] / len(train) * 100
)

missing_values

,missing_count,missing_pct
id,0,0.0
date,0,0.0
store_nbr,0,0.0
family,0,0.0
sales,0,0.0
onpromotion,0,0.0


For sales, we have validated that:

- Business grain     = date × store_nbr × family
- Rows               = 3,000,888
- Stores             = 54
- Families           = 33
- Demand series      = 1,782
- Observed dates     = 1,684
- Calendar gaps      = 4 global dates: on Xmas day from 2013 to 2016 
- Duplicate grain    = 0
- Null values        = 0

The raw sales table has no null values and forms a balanced panel across 1,782 store-family series over 1,684 observed dates. Four calendar dates are globally absent: Christmas Day from 2013 to 2016.
We could reasonably assume that stores are closed (potentially a real signal of null sales/demand) on Christmas Day or there is a dataset convention for Xmas, but `train.csv` alone cannot confirm it. We will keep that in mind when we will look at `holidays_events.csv`.

### `stores.csv`:
Store metadata, including city, state, type, and cluster. A cluster is a grouping of similar stores.

In [13]:
# Load store-level metadata.
stores = pd.read_csv(STORES_PATH)

print("Shape:", stores.shape)
display(stores.head())
display(stores.dtypes.to_frame("dtype"))

Shape: (54, 5)


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


,dtype
store_nbr,int64
city,str
state,str
type,str
cluster,int64


In [14]:
# Validate the expected store-level grain.
print("Unique stores:", stores["store_nbr"].nunique())
print("Duplicate store_nbr:", stores.duplicated(["store_nbr"]).sum())
print("Missing values:")
display(stores.isna().sum().to_frame("missing_count"))

Unique stores: 54
Duplicate store_nbr: 0
Missing values:


,missing_count
store_nbr,0
city,0
state,0
type,0
cluster,0


At this stage, we don't start feature engineering, we just want to determine if **store_nbr** is indeed the table's key to join it to the column with the same name from `train.csv`.

What the variables represent:
- city: store location
- state: region/province
- type: store category
- cluster: group of similar stores

These variables are static features: they describe the store and do not change on a daily basis.

Two important nuances to keep in mind:
1. cluster is stored as an int64, but it is likely not an ordinal numerical variable in the machine learning sense.
   Cluster 13 is not “greater than” cluster 4; rather, it is a categorical identifier.
2. city and state introduce a useful geographic/hierarchical structure.
   Stores in the same city or region may share demand patterns, local holidays, or similar behaviors.
   
For a future global forecasting model, this is precisely the type of metadata that enables information sharing across the 1,782 series.

### `transactions.csv`:
Conceptually, transactions represent the number of customer transactions in one store on one day.

In [15]:
# Load daily store transaction counts.
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    parse_dates=["date"]
)

print("Shape:", transactions.shape)
display(transactions.head())
display(transactions.dtypes.to_frame("dtype"))

Shape: (83488, 3)


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


,dtype
date,datetime64[us]
store_nbr,int64
transactions,int64


In [16]:
# Validate the expected date-store grain.
transaction_grain = ["date", "store_nbr"]

print("Unique stores:", transactions["store_nbr"].nunique())
print("Unique dates:", transactions["date"].nunique())
print(
    "Duplicate date-store rows:",
    transactions.duplicated(transaction_grain).sum()
)

print("Missing values:")
display(transactions.isna().sum().to_frame("missing_count"))

Unique stores: 54
Unique dates: 1682
Duplicate date-store rows: 0
Missing values:


,missing_count
date,0
store_nbr,0
transactions,0


Since this dataset's grain is data x store (and not data x store x family like the Sales data in `train.csv`), this means that the same transaction value would be shared across the 33 product families within a store for a given date after the join operation.
This is potentially a very strong signal for store traffic: the more customers there are in the store, the higher sales tend to be.

However, we need to keep in mind a potential red flag in mind for later:
is the transaction value for the day being forecast actually known at the time the forecast is made?

If the answer is no (and it is!), using the transaction data from that same day could lead to data leakage.

And regarding causality, there is an even more subtle issue: a promotion can itself increase store traffic. Therefore, naively controlling for same-day transactions might eliminate part of the very effect we are trying to measure.

In [17]:
# Inspect the time coverage of transaction data.
print("Start date:", transactions["date"].min().date())
print("End date:", transactions["date"].max().date())

Start date: 2013-01-01
End date: 2017-08-15


In [18]:
# Identify sales dates with no transaction data at all.
missing_transaction_dates = (
    pd.Index(train["date"].unique())
    .difference(transactions["date"].unique())
    .sort_values()
)

print("Sales dates missing from transactions:")
for date in missing_transaction_dates:
    print(date.date())

Sales dates missing from transactions:
2016-01-01
2016-01-03


In [19]:
# Check transaction-history coverage across stores.
transactions_per_store = (
    transactions.groupby("store_nbr")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observed_dates=("date", "nunique"),
    )
)

display(transactions_per_store)

,first_date,last_date,observed_dates
store_nbr,,,
1,2013-01-02,2017-08-15,1676
2,2013-01-02,2017-08-15,1677
3,2013-01-02,2017-08-15,1676
4,2013-01-02,2017-08-15,1676
5,2013-01-02,2017-08-15,1677
6,2013-01-02,2017-08-15,1676
7,2013-01-02,2017-08-15,1675
8,2013-01-02,2017-08-15,1676
9,2013-01-02,2017-08-15,1676


In [20]:
# Summarize how transaction coverage varies across stores.
display(transactions_per_store["observed_dates"].describe())

count      54.000000
mean     1546.074074
std       331.899101
min       118.000000
25%      1659.250000
50%      1676.000000
75%      1677.000000
max      1678.000000
Name: observed_dates, dtype: float64

For transactions, we have validated that:
- Grain                  = date × store_nbr
- Global coverage        = 2013-01-01 → 2017-08-15
- Unique dates           = 1,682
- Dates absent vs sales  = 2016-01-01, 2016-01-03
- Duplicate grain        = 0
- Null values            = 0

### Transactions — initial observations

- Grain: `date × store_nbr`
- No duplicate keys or explicit null values.
- Transaction history is not balanced across stores.
- Several stores start appearing later in the dataset, while others have gaps within their history.
- Missing transaction rows must not be automatically interpreted as zero transactions, therefore we must be cautious with data imputation.
- Same-day transactions may also be unavailable at forecast time, creating a potential leakage risk.

### `oil.csv`:
Daily oil price. Includes values during both the train and test data timeframes. (Ecuador is an oil-dependent country and it's economical health is highly vulnerable to shocks in oil prices.)

In [21]:
# Load daily oil prices.
oil = pd.read_csv(
    OIL_PATH,
    parse_dates=["date"]
)

print("Shape:", oil.shape)
display(oil.head())
display(oil.dtypes.to_frame("dtype"))

Shape: (1218, 2)


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


,dtype
date,datetime64[us]
dcoilwtico,float64


In [22]:
# Inspect the time coverage and uniqueness of oil prices.
print("Start date:", oil["date"].min().date())
print("End date:", oil["date"].max().date())
print("Unique dates:", oil["date"].nunique())
print("Duplicate dates:", oil.duplicated(["date"]).sum())

print("Missing values:")
display(oil.isna().sum().to_frame("missing_count"))

Start date: 2013-01-01
End date: 2017-08-31
Unique dates: 1218
Duplicate dates: 0
Missing values:


,missing_count
date,0
dcoilwtico,43


In [23]:
# Quantify missing oil-price observations.
missing_oil = oil["dcoilwtico"].isna().sum()
missing_oil_pct = missing_oil / len(oil) * 100

print("Missing oil prices:", missing_oil)
print(f"Missing oil prices (%): {missing_oil_pct:.2f}")

Missing oil prices: 43
Missing oil prices (%): 3.53


In [24]:
# Check whether oil observations are present on weekends.
oil_weekdays = oil["date"].dt.day_name()

weekend_rows = oil[
    oil_weekdays.isin(["Saturday", "Sunday"])
]

print("Weekend observations:", len(weekend_rows))
print("\nObserved dates by day of week:")
display(oil_weekdays.value_counts().reindex([
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]))

Weekend observations: 0

Observed dates by day of week:


date
Monday       243.0
Tuesday      244.0
Wednesday    244.0
Thursday     244.0
Friday       243.0
Saturday       NaN
Sunday         NaN
Name: count, dtype: float64

In [25]:
# Compare the oil dataset with a complete daily calendar.
full_oil_calendar = pd.date_range(
    start=oil["date"].min(),
    end=oil["date"].max(),
    freq="D"
)

missing_oil_dates = full_oil_calendar.difference(oil["date"])

missing_dates_by_weekday = (
    pd.Series(missing_oil_dates.day_name())
    .value_counts()
    .reindex([
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday",
    ])
    .fillna(0)
    .astype(int)
)

print("Missing calendar dates:", len(missing_oil_dates))
print("\nMissing dates by day of week:")
display(missing_dates_by_weekday.to_frame("missing_dates"))

Missing calendar dates: 486

Missing dates by day of week:


,missing_dates
Monday,0
Tuesday,0
Wednesday,0
Thursday,0
Friday,0
Saturday,243
Sunday,243


For oil WTI prices, we have validated that:
- Grain                 = date
- Rows                  = 1,218
- Coverage              = 2013-01-01 → 2017-08-31
- Duplicate dates       = 0
- Explicit missing      = 43 oil prices (3.53%)

### Oil prices — initial observations

- Grain: `date`
- No duplicate dates.
- The table contains exactly Monday-to-Friday observations; weekends are absent: we would probably need to make forward-fill since the market prices are closed during the week-end, meaning we will keep the last information available at the time.
- 43 weekday rows contain an explicit missing oil price.
- Oil-price coverage extends beyond the sales training period.
- A daily forecasting dataset will require handling both non-observed weekends and explicit missing prices.
- Oil prices should only be retained later if they provide useful predictive signal and are available at forecast time.

### `holidays_events.csv`:
Holidays and Events, with metadata

- NOTE: Pay special attention to the transferred column. A holiday that is transferred officially falls on that calendar day, but was moved to another date by the government. A transferred day is more like a normal day than a holiday. To find the day that it was actually celebrated, look for the corresponding row where type is Transfer. For example, the holiday Independencia de Guayaquil was transferred from 2012-10-09 to 2012-10-12, which means it was celebrated on 2012-10-12. Days that are type Bridge are extra days that are added to a holiday (e.g., to extend the break across a long weekend). These are frequently made up by the type Work Day which is a day not normally scheduled for work (e.g., Saturday) that is meant to payback the Bridge.
- Additional holidays are days added a regular calendar holiday, for example, as typically happens around Christmas (making Christmas Eve a holiday).

In [26]:
# Load holiday and event metadata.
holidays = pd.read_csv(
    HOLIDAYS_PATH,
    parse_dates=["date"]
)

print("Shape:", holidays.shape)
display(holidays.head(10))
display(holidays.dtypes.to_frame("dtype"))

Shape: (350, 6)


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False


,dtype
date,datetime64[us]
type,str
locale,str
locale_name,str
description,str
transferred,bool


In [27]:
# Inspect time coverage, duplicate dates, and missing values.
print("Start date:", holidays["date"].min().date())
print("End date:", holidays["date"].max().date())
print("Unique dates:", holidays["date"].nunique())
print("Duplicate dates:", holidays.duplicated(["date"]).sum())

print("Missing values:")
display(holidays.isna().sum().to_frame("missing_count"))

Start date: 2012-03-02
End date: 2017-12-26
Unique dates: 312
Duplicate dates: 38
Missing values:


,missing_count
date,0
type,0
locale,0
locale_name,0
description,0
transferred,0


In [28]:
# Inspect the main holiday and event categories.
for column in ["type", "locale", "locale_name", "transferred"]:
    print(f"\n{column}:")
    print(holidays[column].value_counts(dropna=False))


type:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

locale:
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

locale_name:
locale_name
Ecuador                           174
Quito                              13
Riobamba                           12
Guaranda                           12
Latacunga                          12
Ambato                             12
Guayaquil                          11
Cuenca                              7
Ibarra                              7
Manta                               6
Cotopaxi                            6
Libertad                            6
Puyo                                6
Imbabura                            6
Machala                             6
Santo Domingo                       6
El Carmen                           6
Cayambe                             6
Esmeraldas                          6
Quevedo         

In [29]:
# Check whether any complete holiday/event records are duplicated.
print("Fully duplicated rows:", holidays.duplicated().sum())

Fully duplicated rows: 0


In [30]:
# Count how many holiday/event records can occur on the same date.
events_per_date = holidays.groupby("date").size()

print("Maximum records on one date:", events_per_date.max())

display(
    events_per_date
    .sort_values(ascending=False)
    .head(10)
    .to_frame("event_records")
)

Maximum records on one date: 4


,event_records
date,
2014-06-25,4
2017-06-25,3
2016-06-25,3
2015-06-25,3
2013-06-25,3
2012-06-25,3
2014-12-26,2
2015-12-22,2
2012-12-22,2


In [31]:
# Inspect dates associated with multiple holiday/event records.
duplicate_holiday_dates = (
    holidays[
        holidays.duplicated("date", keep=False)
    ]
    .sort_values(["date", "locale", "locale_name", "type"])
)

display(duplicate_holiday_dates.head(30))

,date,type,locale,locale_name,description,transferred
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
11,2012-07-03,Holiday,Local,El Carmen,Cantonizacion de El Carmen,False
10,2012-07-03,Holiday,Local,Santo Domingo,Fundacion de Santo Domingo,False
32,2012-12-22,Holiday,Local,Salinas,Cantonizacion de Salinas,False
33,2012-12-22,Additional,National,Ecuador,Navidad-3,False
36,2012-12-24,Additional,National,Ecuador,Navidad-1,False
35,2012-12-24,Bridge,National,Ecuador,Puente Navidad,False
40,2012-12-31,Additional,National,Ecuador,Primer dia del ano-1,False


The holiday table is event-level rather than date-level. Multiple national, regional or local events may coexist on the same calendar date.

### Holidays and events — initial observations

- The table contains 350 event records across 312 unique dates.
- There are no fully duplicated records.
- Multiple holiday/event records can occur on the same date, with up to four records observed.
- Records can apply at national, regional, or local level.
- Therefore, `date` is not a unique key and a naive date-only join with sales would duplicate observations.
- Holiday/event features will need to be mapped to stores using their geographic scope before joining the sales table.

In [32]:
# Inspect holidays that were officially transferred to another date.
transferred_holidays = holidays[
    holidays["transferred"]
].sort_values("date")

display(transferred_holidays)

,date,type,locale,locale_name,description,transferred
19,2012-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
72,2013-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
135,2014-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
255,2016-05-24,Holiday,National,Ecuador,Batalla de Pichincha,True
266,2016-07-25,Holiday,Local,Guayaquil,Fundacion de Guayaquil,True
268,2016-08-10,Holiday,National,Ecuador,Primer Grito de Independencia,True
297,2017-01-01,Holiday,National,Ecuador,Primer dia del ano,True
303,2017-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,True
312,2017-05-24,Holiday,National,Ecuador,Batalla de Pichincha,True
324,2017-08-10,Holiday,National,Ecuador,Primer Grito de Independencia,True


In [33]:
# Inspect the dates on which transferred holidays were actually observed.
transfer_events = holidays[
    holidays["type"] == "Transfer"
].sort_values("date")

display(transfer_events)

,date,type,locale,locale_name,description,transferred
20,2012-10-12,Transfer,National,Ecuador,Traslado Independencia de Guayaquil,False
73,2013-10-11,Transfer,National,Ecuador,Traslado Independencia de Guayaquil,False
136,2014-10-10,Transfer,National,Ecuador,Traslado Independencia de Guayaquil,False
256,2016-05-27,Transfer,National,Ecuador,Traslado Batalla de Pichincha,False
265,2016-07-24,Transfer,Local,Guayaquil,Traslado Fundacion de Guayaquil,False
269,2016-08-12,Transfer,National,Ecuador,Traslado Primer Grito de Independencia,False
298,2017-01-02,Transfer,National,Ecuador,Traslado Primer dia del ano,False
304,2017-04-13,Transfer,Local,Cuenca,Fundacion de Cuenca,False
313,2017-05-26,Transfer,National,Ecuador,Traslado Batalla de Pichincha,False
325,2017-08-11,Transfer,National,Ecuador,Traslado Primer Grito de Independencia,False


- Holidays marked as `transferred=True` correspond to their original calendar date, not the effective observed holiday.
- The effective replacement date appears separately with `type="Transfer"`.
- Therefore, transferred holidays must not be treated as active holidays on their original date.

## Cross-table integrity checks

In [34]:
# Check whether all sales stores are covered by store metadata.
train_stores = set(train["store_nbr"].unique())
metadata_stores = set(stores["store_nbr"].unique())

print("Sales stores missing from stores.csv:", train_stores - metadata_stores)
print("Metadata stores absent from train.csv:", metadata_stores - train_stores)

Sales stores missing from stores.csv: set()
Metadata stores absent from train.csv: set()


In [35]:
# Check whether transaction stores are covered by store metadata.
transaction_stores = set(transactions["store_nbr"].unique())

print(
    "Transaction stores missing from stores.csv:",
    transaction_stores - metadata_stores
)

Transaction stores missing from stores.csv: set()


In [36]:
# Check whether local and regional holiday locations match store metadata.
local_holiday_locations = set(
    holidays.loc[holidays["locale"] == "Local", "locale_name"]
)

regional_holiday_locations = set(
    holidays.loc[holidays["locale"] == "Regional", "locale_name"]
)

store_cities = set(stores["city"])
store_states = set(stores["state"])

print(
    "Local holiday locations without matching store city:",
    local_holiday_locations - store_cities
)

print(
    "Regional holiday locations without matching store state:",
    regional_holiday_locations - store_states
)

Local holiday locations without matching store city: set()
Regional holiday locations without matching store state: set()


## Raw data exploration — summary

### Sales
- Grain: `date × store_nbr × family`
- 3,000,888 rows across 54 stores and 33 product families.
- 1,782 complete store-family series.
- 1,684 observed dates from 2013-01-01 to 2017-08-15.
- Four calendar dates are globally absent: Christmas Day from 2013 to 2016.
- No duplicate business keys or explicit missing values.

### Stores
- Grain: `store_nbr`
- One metadata record per store.
- Contains geographic and store-level categorical attributes.
- No duplicates or missing values.

### Transactions
- Grain: `date × store_nbr`
- No duplicate keys or explicit missing values.
- Coverage is not balanced across stores.
- Some stores appear later in the history and some have missing transaction dates.
- Missing rows must not be interpreted automatically as zero transactions.

### Oil prices
- Grain: `date`
- Exactly Monday-to-Friday observations; weekends are absent.
- 43 weekday rows contain an explicit missing oil price.
- No duplicate dates.
- A daily feature dataset will require handling both absent weekends and explicit missing values.

### Holidays and events
- Event-level table rather than date-level.
- Multiple events can occur on the same date.
- Events apply at national, regional, or local geographic scope.
- Transferred holidays are represented by the original date with `transferred=True` and a separate effective `Transfer` record.
- A naive date-only join would duplicate sales observations.

### Cross-table integrity
- Store identifiers are consistent across sales, store metadata, and transactions.
- Local holiday locations match store cities.
- Regional holiday locations match store states.

## Data Science business case layer
**A retailer gives you 1,782 demand series. What do you ask for before discussing a model?**

Answer: **Business problem → target/grain → history/quality → demand drivers → hierarchy → forecast-time availability.**

And we can attach Favorita Store Sales data near perfectly to this grid:
- `train.csv` → target, grain, history, completeness
- `stores.csv` → static metadata + hierarchy
- `transactions.csv` → dynamic demand/activity signal
- `holidays_events.csv` → known calendar effects
- `oil.csv` → external macro signal
But the real company dataset would also have prices, promotions and stocks.

Before discussing a model, I’d first clarify what the 1,782 series represent and what business decision the forecasts will support: the forecasting horizon, frequency and required level of granularity.
Then I’d look at the target history itself: how much history is available, whether all series have comparable coverage, whether there are missing dates or structural zeros, and whether the history is long enough to observe weekly or yearly seasonality.
I’d also ask for the main demand drivers. In retail, that would typically include prices, promotions, inventory and stock-outs, holidays and events, as well as dynamic signals such as store traffic or transactions and relevant external variables.
I’d want static metadata as well — store location, store type, product category and any hierarchy between stores and products — because information can potentially be shared across related series.
Finally, I’d check which of those variables are actually known at forecast time. A feature can be predictive historically but unusable operationally if it isn’t available when the forecast is produced.
Only once I understand the forecasting objective, the series structure, data quality and the information available at prediction time would I start discussing model families.